## Sample the data for inspection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, confusion_matrix,
    classification_report
)

## Data Loading

In [ ]:
TRAIN_PATH = "../../data/transformed/train_t03.parquet"
VAL_PATH   = "../../data/transformed/val_t03.parquet"
TEST_PATH  = "../../data/transformed/test_t03.parquet"

TARGET   = "Results"

train = pd.read_parquet(TRAIN_PATH)
val   = pd.read_parquet(VAL_PATH)
test  = pd.read_parquet(TEST_PATH)

FEATURES = [col for col in train.columns if col != TARGET]

X_train, y_train = train[FEATURES], train[TARGET]
X_val,   y_val   = val[FEATURES],   val[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"\nTarget distribution (train):\n{y_train.value_counts()}")

## Hyperparameters

In [ ]:
params = {
    "n_estimators":      200,
    "max_depth":         10,
    "min_samples_split": 5,
    "min_samples_leaf":  2,
    "max_features":      "sqrt",
    "class_weight":      "balanced",
    "random_state":      42,
    "n_jobs":            -1,
}

## Training

In [ ]:
model = RandomForestClassifier(**params)
model.fit(X_train, y_train)

print("Model trained successfully.")
print(f"Number of trees: {model.n_estimators}")

## Validation

In [ ]:
y_val_pred = model.predict(X_val)
y_val_prob = model.predict_proba(X_val)[:, 1]

val_cm = confusion_matrix(y_val, y_val_pred)
tn, fp, fn, tp = val_cm.ravel()

val_metrics = {
    # Standard metrics
    "val_accuracy":  accuracy_score(y_val, y_val_pred),
    "val_f1":        f1_score(y_val, y_val_pred),
    "val_roc_auc":   roc_auc_score(y_val, y_val_prob),
    "val_precision": precision_score(y_val, y_val_pred),
    "val_recall":    recall_score(y_val, y_val_pred),
    # Business metrics
    # Restaurants wrongly predicted to pass but actually fail = public health risk
    "val_false_negative_rate":        fn / (fn + tp),
    # Of all failing restaurants, how many did we correctly flag?
    "val_failing_catch_rate":         tp / (tp + fn),
}

print("=== VALIDATION METRICS ===")
for k, v in val_metrics.items():
    print(f"  {k:<35} {v:.4f}")

print(f"\nClassification Report (Val):\n")
print(classification_report(y_val, y_val_pred, target_names=["Pass (0)", "Fail (1)"]))

## Testing

In [ ]:
y_test_pred = model.predict(X_test)
y_test_prob = model.predict_proba(X_test)[:, 1]

test_cm = confusion_matrix(y_test, y_test_pred)
tn, fp, fn, tp = test_cm.ravel()

test_metrics = {
    # Standard metrics
    "test_accuracy":  accuracy_score(y_test, y_test_pred),
    "test_f1":        f1_score(y_test, y_test_pred),
    "test_roc_auc":   roc_auc_score(y_test, y_test_prob),
    "test_precision": precision_score(y_test, y_test_pred),
    "test_recall":    recall_score(y_test, y_test_pred),
    # Business metrics
    "test_false_negative_rate":  fn / (fn + tp),
    "test_failing_catch_rate":   tp / (tp + fn),
}

print("=== TEST METRICS ===")
for k, v in test_metrics.items():
    print(f"  {k:<35} {v:.4f}")

print(f"\nClassification Report (Test):\n")
print(classification_report(y_test, y_test_pred, target_names=["Pass (0)", "Fail (1)"]))

## Confusion Matrix Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, cm, title in zip(
    axes,
    [val_cm, test_cm],
    ["Validation Set", "Test Set"]
):
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Pass (0)", "Fail (1)"]
    )
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"Confusion Matrix — {title}", fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Predicted Label", fontsize=11)
    ax.set_ylabel("True Label", fontsize=11)

plt.suptitle("Random Forest — Confusion Matrices", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("confusion_matrix_rf.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved as confusion_matrix_rf.png")

## Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    "feature":    FEATURES,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("=== FEATURE IMPORTANCES ===")
print(importance_df.to_string(index=False))

## Exporting To MLflow

In [ ]:
# Just verify everything looks correct before handing off.

rf_output = {
    "model":        model,
    "params":       params,
    "val_metrics":  val_metrics,
    "test_metrics": test_metrics,
    "features":     FEATURES,
    "importance":   importance_df,
}

print("Random Forest output ready for experiment_tracking.py")
print(f"Val F1:  {val_metrics['val_f1']:.4f}")
print(f"Test F1: {test_metrics['test_f1']:.4f}")